# 08. 대형 언어 모델(LLM) 활용

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_08_%EB%8C%80%ED%98%95_%EC%96%B8%EC%96%B4_%EB%AA%A8%EB%8D%B8_LLM_%ED%99%9C%EC%9A%A9.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy", "transformers", "peft"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 실습: 미니 인과적 트랜스포머와 RAG

### 🔬 실습 — Scaled Dot-Product Attention과 미니 트랜스포머

① **Scaled Dot-Product Attention**: 인과적 마스킹 포함.


In [ ]:
import torch, torch.nn.functional as F, math

def attention(Q, K, V, causal=True):
    # Q,K,V: (B, T, d_k)
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)   # (B,T,T)
    if causal:
        T = scores.size(-1)
        mask = torch.triu(torch.ones(T, T, device=scores.device),
                          diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf")) # 미래 차단
    attn = F.softmax(scores, dim=-1)
    return attn @ V                                      # (B,T,d_k)

Q = K = V = torch.randn(2, 16, 64)          # (B, T, d_k) 동작 확인
print("attention output shape:", attention(Q, K, V).shape)

**실행 결과 예시**

```
attention output shape: torch.Size([2, 16, 64])
```

차원이 의도대로 유지되는지 확인한다. 출력 모양은 입력과 같은 \((B, T, d_k)\)이고, 인과적 마스크 덕분에 각 위치 \(t\)는 \(t\) 이하만 참조한다.


② **멀티헤드 어텐션**: \(H\)개 헤드로 나눠 병렬로 계산한 뒤 결합한다.


In [ ]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d, H):
        super().__init__()
        self.H, self.dk = H, d // H
        self.qkv = nn.Linear(d, 3 * d); self.out = nn.Linear(d, d)
    def forward(self, x):                       # x: (B,T,d)
        B, T, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.H, self.dk).transpose(1, 2)
        q, k, v = map(split, (q, k, v))         # (B,H,T,dk)
        o = attention(q, k, v)                  # 헤드별 어텐션
        o = o.transpose(1, 2).reshape(B, T, d)  # 결합
        return self.out(o)

③ **트랜스포머 블록**: Pre-LN + 잔차 + FFN이다.


In [ ]:
class Block(nn.Module):
    def __init__(self, d, H):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = MultiHeadAttention(d, H)
        self.ffn = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(),
                                 nn.Linear(4*d, d))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))          # Pre-LN + 잔차
        x = x + self.ffn(self.ln2(x))
        return x

④ **생성(자기회귀 + 온도 샘플링)**: 다음 토큰을 반복 예측한다.


In [ ]:
@torch.no_grad()
def generate(model, idx, max_new, temperature=1.0):
    for _ in range(max_new):
        logits = model(idx)[:, -1, :] / temperature    # 마지막 위치
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1)              # 샘플링
        idx = torch.cat([idx, nxt], dim=1)
    return idx

### 🔬 실습(노트북 보강) — LoRA로 BERT 분류기 파인튜닝

8.3절의 LoRA를 7장 실습과 같은 klue/bert-base 분류 문제에 적용한다. 본문에는 없는 노트북 보강 실습이다.

① **LoRA 어댑터 부착.** 어텐션의 \(W_Q, W_V\)에만 랭크 \(r=8\)의 \(A, B\) 행렬을 달고 본체는 동결한다.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

torch.manual_seed(0)                                   # 재현성(2장)
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")
base = AutoModelForSequenceClassification.from_pretrained("klue/bert-base", num_labels=2)
cfg = LoraConfig(r=8, lora_alpha=16,                   # 랭크 r, 스케일 alpha/r = 2
                 target_modules=["query", "value"],    # 어텐션의 W_Q, W_V에만 어댑터
                 lora_dropout=0.0, task_type="SEQ_CLS")
model = get_peft_model(base, cfg)                      # 본체 동결 + A, B 행렬 추가
model.print_trainable_parameters()

② **어댑터만 학습하고 병합.** 학습할 수 있는 파라미터(어댑터와 분류 헤드)만 옵티마이저에 넘기고, 학습이 끝나면 \(W+\frac{\alpha}{r}BA\)로 병합해 추가 지연 없는 단일 모델로 만든다.

In [ ]:
texts  = ["K2 소총의 정비 주기는 사격 100발 이후이다.",
          "전방 관측소에서 적 병력 이동이 탐지되었다."]
labels = torch.tensor([0, 1])                          # 0: 정비, 1: 정보보고
enc = tokenizer(texts, padding="max_length", truncation=True,
                max_length=64, return_tensors="pt")
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-3)
model.train()
for epoch in range(3):
    out = model(**enc, labels=labels)
    out.loss.backward(); optimizer.step(); optimizer.zero_grad()
    print(f"ep{epoch+1} loss {out.loss.item():.4f}")

merged = model.merge_and_unload()                      # W <- W + (alpha/r) B A 로 병합
print("병합 후 LoRA 모듈 수:", sum("lora" in n for n, _ in merged.named_modules()))

**실행 결과 예시**

```
trainable params: 296,450 || all params: 110,915,332 || trainable%: 0.2673
ep1 loss 0.8322
ep2 loss 0.5442
ep3 loss 0.4284
병합 후 LoRA 모듈 수: 0
```

(코드를 CPU에서 시드 0으로 그대로 실행한 결과이다. 학습하는 파라미터는 12개 층 × 2곳 × \((8\times768+768\times8)=294{,}912\)개의 어댑터와 분류 헤드 1,538개로, 전체의 0.27%뿐이다. 7장의 전체 파인튜닝과 달리 본체 약 1.1억 개는 그대로이므로, 임무별로 이 작은 어댑터만 바꿔 끼울 수 있다(8.3절). 두 문장 학습은 흐름을 보이기 위한 장난감 예이다.)

### 🔬 실습 — 최소 RAG 파이프라인

① **문서 임베딩과 인덱싱**: 청크를 문장 임베딩으로 변환한다.


In [ ]:
# sentence-transformers는 setup에 없으므로 필요 시 설치(최초 1회, 모델 다운로드에 인터넷 필요)
import importlib.util, subprocess, sys
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
from sentence_transformers import SentenceTransformer
import torch, torch.nn.functional as F

embedder = SentenceTransformer("jhgan/ko-sroberta-multitask")  # 한국어
docs = ["K2 소총 정비 주기는 사격 100발 이후이다.",
        "K9 자주포의 최대 사거리는 40km이다.",
        "야간 정찰 시 열영상 장비를 우선 사용한다."]
doc_emb = torch.tensor(embedder.encode(docs))                  # (3, d)

② **질의 검색**: 코사인 유사도로 Top-\(k\) 청크를 선택한다.


In [ ]:
def retrieve(query, k=2):
    q = torch.tensor(embedder.encode([query]))                  # (1, d)
    sims = F.cosine_similarity(q, doc_emb)                      # (3,)
    topk = sims.topk(k).indices
    return [docs[i] for i in topk]

ctx = retrieve("소총 정비는 얼마마다 하나?")
print(ctx)

**실행 결과 예시**

```
['K2 소총 정비 주기는 사격 100발 이후이다.', 'K9 자주포의 최대 사거리는 40km이다.']
```

(본문 코드를 그대로 실행한 검색 결과이다. 정답 문서가 1위로 검색되었고(세 문서와의 코사인 유사도 0.584/0.298/0.222), k=2라 차순위 문서가 함께 담겼다. 아래 프롬프트에 "<context>만 근거로"라는 지시가 있으므로 무관한 차순위가 섞여도 답은 1위 근거에서 나온다. 이 프롬프트를 LLM에 넣으면 "사격 100발 이후 정비합니다(출처: 1번 문서)"류의 근거 기반 답변을 얻는다.)


③ **프롬프트 구성**: 검색 청크를 컨텍스트로 주입한다(인젝션 방어 구분자 포함).


In [ ]:
prompt = f"""당신은 군사 기술 보조원입니다. 아래 <context>만 근거로 답하고,
근거가 없으면 '자료 없음'이라 답하십시오.
<context>
{chr(10).join(ctx)}
</context>
질문: 소총 정비는 얼마마다 하나?"""

④ **(노트북 보강) 소형 LLM으로 답 생성.** 미국 IBM이 공개한 granite-4.0-350m(Apache-2.0, 약 0.7GB, 한국어 지원)에 위 프롬프트를 넣어 답을 생성한다. 실행마다 같은 답이 나오도록 탐욕 디코딩을 쓴다. 문서에 근거가 없는 질문도 하나 던져 '자료 없음' 지시를 지키는지 본다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
name = "ibm-granite/granite-4.0-350m"           # IBM(미국), Apache-2.0, 약 0.7GB
tok = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name)

def answer(p, max_new_tokens=60):
    enc = tok.apply_chat_template([{"role": "user", "content": p}],
                                  add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens,
                           do_sample=False)    # 탐욕 디코딩: 실행마다 같은 답
    n = enc["input_ids"].shape[1]                # 프롬프트 길이
    return tok.decode(out[0, n:], skip_special_tokens=True).strip()

print("답 1:", answer(prompt))
q2 = "K2 소총의 유효 사거리는?"                 # 문서에 근거가 없는 질문
p2 = prompt.replace("소총 정비는 얼마마다 하나?", q2).replace(
    chr(10).join(ctx), chr(10).join(retrieve(q2)))
print("답 2:", answer(p2))

**실행 결과 예시**

```
답 1: 소총 정비는 사격 100발 이후이다.
답 2: K2 소총의 유효 사거리는 100발입니다.
```

(CPU에서 그대로 실행한 결과이다. 답 1은 1위 근거를 정확히 옮겼다. 답 2는 문서에 근거가 없는 질문인데 '자료 없음' 대신 근거 문장의 '100발'을 사거리로 바꿔 지어냈다. 3.5억 파라미터의 작은 모델은 지시를 끝까지 지키지 못하는 일이 잦고, 이런 답은 그럴듯해 보인다. 8.6절처럼 답의 주장을 근거 문서와 하나씩 대조하는 검사 없이 그대로 쓰면 안 되는 이유이다.)

---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
